# Complete PWAssist Walkthrough
Here we will walk through all the capabilities of the library by running a complete analysis step-by-step. This is intended to be a verbose example explaining every stage of the process, but many of the steps are collected in a simple pipeline that does the process for you. See the [quick start]() page as an example (TODO: create quick start page)

## Requirements
If you haven't already, please read the [README.md](../../README.md) file on the repo's home page to understand at what analysis stage this library will be useful

Let's quickly define a couple of useful constants for this notebook

In [ ]:
import pathlib
DATA_DIR = pathlib.Path("../data/")

## Collecting Data
We will be using sets of fit results from three independent mass bins collected in a [data directory](../data/). Let's print these files first

In [ ]:
for path in DATA_DIR.iterdir():
    if path.is_dir():
        print(f"Directory: {path.name}")
        for file in path.iterdir():
            if file.is_file():
                print(f"  File: {file.name}")

The first step is to [build a catalog](../../src/pwassist/io/catalog.py). A catalog consists of a *manifest* that itemizes all relevant fit result files in each mass bin. To build it, we first scan the directories, which identifies the various fit result types (fit, data, covariance, etc.) and their metadata.

In [ ]:
from pwassist.io.catalog import Catalog
cat = Catalog(DATA_DIR)

example_bin = DATA_DIR / "mass_1.54-1.56/"

for file in example_bin.iterdir():
    if file.is_file() and file.suffix == ".csv":
        f_type = cat.identify_file_type(file)

        print(f"File: {file.name}, \tType: {f_type}")

In [ ]:
cat.scan()
cat.manifest.head(10)

With a catalog created, we can then move on to bundling these results per bin into [BinBundles and BinCollections](../../src/pwassist/io/binning.py). These primarily serve as convenient classes for our later preprocessor to iterate on

In [ ]:
from pwassist.io.binning import MassBin, BinBundle, BinCollection

collection = BinCollection.from_catalog(cat)

for mass_bin, bundle in collection:
    print(f"Mass Bin: {mass_bin}, Bundle ID: {bundle.bin_id}")
    for file_type, path in bundle.paths.items():
        print(f"  File Type: {file_type}, Path: {path}")

bundle = collection[MassBin(1.54, 1.56)]

In [ ]:
bundle.fit.frame.head()

In [ ]:
bundle.data.frame.head()

In [ ]:
bundle.unload() # getting data frames from a bundle loads them into memory, so it's good practice to unload them when you're done to free up memory

## Preprocessing
Before we plot, we need to perform a series of checks and operations to prepare the data. These include for example:
- Wrapping phase differences to [-pi, pi) and converting to degrees
- Ensuring the correlation matrices are symmetric and bounded from [-1,1]
Lets print out the default ones


In [ ]:
from pwassist.preprocessing.preprocessor import Preprocessor, ProcessedBin

prep = Preprocessor()
print(prep.step_names)

proc_bin: ProcessedBin = prep.run(bundle)

for step, time in proc_bin.report.timings_ms.items():
    print(f"Step: {step},\tTime: {time:.2f} ms")

for warning in proc_bin.report.warnings:
    print(warning)

Details of the default steps [can be found here](../../src/pwassist/preprocessing/steps.py)

### Creating your own steps